In [1]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

In [2]:
# DB 연결 설정
user = "root"
password = quote_plus("Child119@")
host = "34.47.74.230"
db = "test"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:3306/{db}?charset=utf8mb4")


In [3]:
with engine.connect() as conn:
    # 1️⃣ emergency_geom 테이블 생성 (emergency를 복사)
    conn.execute(text("DROP TABLE IF EXISTS emergency_geom;"))
    conn.execute(text("CREATE TABLE emergency_geom LIKE emergency;"))
    conn.execute(text("INSERT INTO emergency_geom SELECT * FROM emergency;"))

    # 2️⃣ POINT 타입 geometry 컬럼 추가 및 값 채우기
    conn.execute(text("ALTER TABLE emergency_geom ADD COLUMN geom POINT;"))

    conn.execute(text("""
        UPDATE emergency_geom
        SET geom = ST_GeomFromText(CONCAT('POINT(', longitude, ' ', latitude, ')'));
    """))

    # 3️⃣ 공간 인덱스 생성
    conn.execute(text("CREATE SPATIAL INDEX idx_geom ON emergency_geom (geom);"))

    print("✅ 'emergency_geom' 테이블이 생성되었습니다.")

OperationalError: (pymysql.err.OperationalError) (1252, 'All parts of a SPATIAL index must be NOT NULL')
[SQL: CREATE SPATIAL INDEX idx_geom ON emergency_geom (geom);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)